In [1]:
!pip install -q langchain-core langchain-mistralai pandas

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 548.1/548.1 kB 33.4 MB/s eta 0:00:00


In [2]:
import pandas as pd

from langchain_core.prompts import PromptTemplate
from langchain_mistralai import ChatMistralAI

print("Imports correctos")

Imports correctos


Subir dataset

In [3]:
from google.colab import files

uploaded = files.upload()

Saving dataset_rag.csv to dataset_rag.csv


Cargar dataset

In [4]:
df = pd.read_csv("dataset_rag.csv")


Limitar filas (el rendimiento de este varia segun los tokens, pero estos mismos son limitados si son usados de forma gratuita como este caso)

In [29]:
df = df.head(5)

Crear corpus

In [30]:
corpus = ""

for index, row in df.iterrows():

    fila = []

    for col in df.columns:
        fila.append(f"{col}: {row[col]}")

    texto = ", ".join(fila)

    corpus += texto + "\n"

Verificar corpus

In [31]:
print(corpus[:3000])

ORDERNUMBER: -1.6479470949959203, QUANTITYORDERED: -0.5228908557285186, PRICEEACH: 0.5969775019778834, ORDERLINENUMBER: -1.0570587063287502, SALES: -0.3708252319213149, QTR_ID: -1.4270386327380036, MONTH_ID: -1.3929088876103477, YEAR_ID: -1.1651700861775494, MSRP: -0.1422458441233342, ORDER_YEAR: 2003, ORDER_MONTH: 2, ORDER_DAY: 24, STATUS_Cancelled: False, STATUS_Disputed: False, STATUS_In Process: False, STATUS_On Hold: False, STATUS_Resolved: False, STATUS_Shipped: True, PRODUCTLINE_Classic Cars: False, PRODUCTLINE_Motorcycles: True, PRODUCTLINE_Planes: False, PRODUCTLINE_Ships: False, PRODUCTLINE_Trains: False, PRODUCTLINE_Trucks and Buses: False, PRODUCTLINE_Vintage Cars: False, PRODUCTCODE_S10_1678: True, PRODUCTCODE_S10_1949: False, PRODUCTCODE_S10_2016: False, PRODUCTCODE_S10_4698: False, PRODUCTCODE_S10_4757: False, PRODUCTCODE_S10_4962: False, PRODUCTCODE_S12_1099: False, PRODUCTCODE_S12_1108: False, PRODUCTCODE_S12_1666: False, PRODUCTCODE_S12_2823: False, PRODUCTCODE_S12_31

API KEY

In [32]:
import os

os.environ["MISTRAL_API_KEY"] = "Q57RYnh6PD9aYxcMiO3IiT8SJGkcDUpC"

Crear modelo

In [33]:
llm = ChatMistralAI(
    model="mistral-small-latest",
    temperature=0
)

Prompt

In [37]:
template = """
Eres un asistente especializado en análisis de datos empresariales.

Tu tarea es responder preguntas utilizando EXCLUSIVAMENTE
la información proporcionada en el contexto.

REGLAS:
- No inventes información.
- No uses conocimiento externo.
- Responde solamente usando el contexto.
- Si la información no está disponible:
"No encontré esa información en el dataset."
- Sé claro, preciso y breve.

CONTEXTO:
{context}

PREGUNTA:
{question}

RESPUESTA:
"""

Crear agente

In [38]:
prompt = PromptTemplate.from_template(template)

agent = prompt | llm

Primera consulta

In [36]:
respuesta = agent.invoke({
    "context": corpus,
    "question": "¿Qué años aparecen en el dataset?"
})

print(respuesta.content)

2003


segunda consulta

In [39]:
respuesta = agent.invoke({
    "context": corpus,
    "question": "Menciona todos los años diferentes que aparecen en el dataset."
})

print(respuesta.content)

2003


Creacion de RAG

INSTALAR LIBRERÍAS NECESARIAS

In [40]:
!pip install -q \
langchain-core \
langchain-community \
langchain-mistralai \
faiss-cpu \
sentence-transformers \
pandas

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 87.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 77.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 66.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 4.7 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.


SUBIR DATASET(volvemos a subir por si acaso y para llevar bien el progreso)

In [41]:
from google.colab import files

uploaded = files.upload()

Saving dataset_rag.csv to dataset_rag (1).csv


IMPORTAR LIBRERÍAS

In [42]:
import pandas as pd
import os

from langchain_core.prompts import PromptTemplate
from langchain_mistralai import ChatMistralAI

from langchain_community.vectorstores import FAISS
from langchain_community.embeddings import HuggingFaceEmbeddings

/tmp/ipykernel_2904/1121397516.py:7: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import FAISS


CARGAR DATASET

In [43]:
df = pd.read_csv("dataset_rag.csv")

LIMITAR FILAS TEMPORALMENTE

In [64]:
df = df.head(5)

VERIFICAR DATASET

In [65]:
print(df.shape)

df.head()

(5, 726)


,ORDERNUMBER,QUANTITYORDERED,PRICEEACH,ORDERLINENUMBER,SALES,QTR_ID,MONTH_ID,YEAR_ID,MSRP,ORDER_YEAR,...,CONTACTFIRSTNAME_Veysel,CONTACTFIRSTNAME_Victoria,CONTACTFIRSTNAME_Violeta,CONTACTFIRSTNAME_Wendy,CONTACTFIRSTNAME_William,CONTACTFIRSTNAME_Wing C,CONTACTFIRSTNAME_Yoshi,DEALSIZE_Large,DEALSIZE_Medium,DEALSIZE_Small
0,-1.647947,-0.522891,0.596978,-1.057059,-0.370825,-1.427039,-1.392909,-1.16517,-0.142246,2003,...,False,False,False,False,False,False,False,False,False,True
1,-1.495888,-0.112201,-0.114450,-0.347015,-0.427897,-0.596243,-0.572337,-1.16517,-0.142246,2003,...,False,False,False,False,False,False,False,False,False,True
2,-1.354689,0.606505,0.549384,-1.057059,0.179443,0.234553,-0.025289,-1.16517,-0.142246,2003,...,False,False,False,False,False,False,False,False,True,False
3,-1.235214,1.017195,-0.019759,-0.110334,0.104701,0.234553,0.248235,-1.16517,-0.142246,2003,...,False,False,False,False,False,False,False,False,True,False
4,-1.083154,1.427884,0.810158,1.783116,0.896740,1.065350,0.795284,-1.16517,-0.142246,2003,...,False,False,False,False,False,False,False,False,True,False


CONVERTIR DATASET EN DOCUMENTOS DE TEXTO

In [66]:
documents = []

for index, row in df.iterrows():

    fila = []

    for col in df.columns:
        fila.append(f"{col}: {row[col]}")

    texto = ", ".join(fila)

    documents.append(texto)

VERIFICAR DOCUMENTOS

In [67]:
print(documents[0])

print("\nCantidad de documentos:")

print(len(documents))

ORDERNUMBER: -1.6479470949959203, QUANTITYORDERED: -0.5228908557285186, PRICEEACH: 0.5969775019778834, ORDERLINENUMBER: -1.0570587063287502, SALES: -0.3708252319213149, QTR_ID: -1.4270386327380036, MONTH_ID: -1.3929088876103477, YEAR_ID: -1.1651700861775494, MSRP: -0.1422458441233342, ORDER_YEAR: 2003, ORDER_MONTH: 2, ORDER_DAY: 24, STATUS_Cancelled: False, STATUS_Disputed: False, STATUS_In Process: False, STATUS_On Hold: False, STATUS_Resolved: False, STATUS_Shipped: True, PRODUCTLINE_Classic Cars: False, PRODUCTLINE_Motorcycles: True, PRODUCTLINE_Planes: False, PRODUCTLINE_Ships: False, PRODUCTLINE_Trains: False, PRODUCTLINE_Trucks and Buses: False, PRODUCTLINE_Vintage Cars: False, PRODUCTCODE_S10_1678: True, PRODUCTCODE_S10_1949: False, PRODUCTCODE_S10_2016: False, PRODUCTCODE_S10_4698: False, PRODUCTCODE_S10_4757: False, PRODUCTCODE_S10_4962: False, PRODUCTCODE_S12_1099: False, PRODUCTCODE_S12_1108: False, PRODUCTCODE_S12_1666: False, PRODUCTCODE_S12_2823: False, PRODUCTCODE_S12_31

CREAR EMBEDDINGS

In [68]:
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


CREAR VECTOR STORE CON FAISS

In [69]:
vectorstore = FAISS.from_texts(
    documents,
    embeddings
)

CREAR RETRIEVER

In [70]:
retriever = vectorstore.as_retriever(
    search_kwargs={"k": 3}
)

CONFIGURAR API KEY DE MISTRAL

In [92]:
os.environ["MISTRAL_API_KEY"] = "VRTDBMnjUg29Tp7oGL2Y2Fpfy0cb14Eh"

CREAR MODELO MISTRAL

In [93]:
llm = ChatMistralAI(
    model="mistral-small-latest",
    temperature=0
)

CREAR PROMPT DEL RAG

In [94]:
template = """
Eres un asistente especializado en análisis de datos empresariales.

Tu tarea es responder preguntas utilizando EXCLUSIVAMENTE
la información proporcionada en el contexto.

REGLAS IMPORTANTES:
- No inventes información.
- No uses conocimiento externo.
- Responde solamente usando el contexto.
- Si no encuentras la respuesta:
"No encontré esa información en el dataset."
- Sé claro y preciso.

CONTEXTO:
{context}

PREGUNTA:
{question}

RESPUESTA:
"""

CREAR PROMPT TEMPLATE

In [95]:
prompt = PromptTemplate.from_template(template)

CREAR CADENA RAG

In [96]:
rag_chain = prompt | llm

HACER UNA PREGUNTA

In [85]:
question = "¿Qué países aparecen en el dataset?"

RECUPERAR DOCUMENTOS RELEVANTES

In [86]:
retrieved_docs = retriever.invoke(question)

UNIR CONTEXTO RECUPERADO

In [87]:
context = "\n".join(
    [doc.page_content for doc in retrieved_docs]
)

GENERAR RESPUESTA DEL RAG

In [88]:
response = rag_chain.invoke({
    "context": context,
    "question": question
})

MOSTRAR RESPUESTA FINAL

In [89]:
print(response.content)

Australia, Austria, Belgium, Canada, Denmark, Finland, France, Germany, Ireland, Italy, Japan, Norway, Philippines, Singapore, Spain, Sweden, Switzerland, UK, USA


CONSULTAS EXTRA DE PRUEBA
Consulta sobre productos

In [90]:
question = "¿Cuál es el producto más vendido?"

retrieved_docs = retriever.invoke(question)

context = "\n".join(
    [doc.page_content for doc in retrieved_docs]
)

response = rag_chain.invoke({
    "context": context,
    "question": question
})

print(response.content)

No encontré esa información en el dataset.
